##### Module Imports

In [ ]:
import pandas as pd
from xgboost import XGBClassifier, plot_tree
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

import matplotlib.pyplot as pyplot

import sys                                              # For importing custom modules (from parent dirs)
sys.path.append('..')
from tools import serialTools, captureTools, eval

##### Global Vars

In [ ]:
datasetsPath = '../datasets/har/'

In [ ]:
xtrain =    pd.DataFrame()
xtest =     pd.DataFrame()
ytrain =    pd.DataFrame()
ytest =     pd.DataFrame()
bestIter = 0
initialAcc = 0

leGender = LabelEncoder()       # Female, Male
leHML = LabelEncoder()          # High, Moderate, Low
leChestPain = LabelEncoder()    # Non-anginal, Asymptomatic, Typical, Atypical
leThalassemia = LabelEncoder()  # Normal, Fixed Defect, Reversible Defect
leECG = LabelEncoder()          # Normal, ST-T abnormality, Left ventricular hypertrophy

##### Func: Import Data

In [ ]:
def importData():
    print(f'Importing data...')

    global xtrain, xtest, ytrain, ytest

    # Load dataset
    data = pd.read_csv(datasetsPath + 'heart_attack_risk_dataset.csv')

    # Encode Categorical Data
    data['Gender'] = leGender.fit_transform(data['Gender'])
    data['Physical_Activity_Level'] = leHML.fit_transform(data['Physical_Activity_Level'])
    data['Stress_Level'] = leHML.fit_transform(data['Stress_Level'])
    data['Heart_Attack_Risk'] = leHML.fit_transform(data['Heart_Attack_Risk'])
    data['Chest_Pain_Type'] = leChestPain.fit_transform(data['Chest_Pain_Type'])
    data['Thalassemia'] = leThalassemia.fit_transform(data['Thalassemia'])
    data['ECG_Results'] = leECG.fit_transform(data['ECG_Results'])

    # Splitting data into features and labels
    xdata = data.iloc[:,:19]
    ydata = data.iloc[:,19:]

    # Splitting data into training and testing
    xtrain, xtest, ytrain, ytest = train_test_split(
        xdata, 
        ydata, 
        test_size=0.2,
        random_state=0
    )
    print(f'Data imported!')

##### Func: Train Model

In [ ]:
def trainModel(model: XGBClassifier, feats: pd.DataFrame, labels: pd.DataFrame, maxDepth: int = 6, setBestIter: bool = False, evalset: list = None):
    
    global bestIter
    
    if setBestIter == True:
        print(f'\tTraining model (to determine best iteration)...')
        model.set_params(
            objective='multi:softmax',
            num_class=3,
            learning_rate=0.1,
            n_estimators=10000,
            early_stopping_rounds=10,
            max_depth=maxDepth
        )
        model.fit(
            feats, labels,
            eval_set = evalset,
            verbose = False
        )
        bestIter = model.best_iteration
        print(f'\tBest iteration: {bestIter}')
    else:
        print(f'\tTraining model...')
        if bestIter == 0:
            print('BestIter = 0 -> Something is wrong!')
        model.set_params(
            objective='multi:softmax',
            num_class=3,
            learning_rate=0.1,
            n_estimators=bestIter,
            early_stopping_rounds=None,
            max_depth=maxDepth
        )
        model.fit(feats,labels)
        print(f'\tModel trained!')

    

##### Func: Train Quicksave

In [ ]:
def trainQuicksave(maxDepth: int = 6):
    print("Training Quicksave...")
    model = XGBClassifier()
    evalset = [(xtrain,ytrain),(xtest,ytest)]
    trainModel(model, xtrain, ytrain, maxDepth,True, evalset)
    trainModel(model, xtrain, ytrain, maxDepth)
    model.save_model("quicksave.json")
    print("Quicksave model trained and saved as quicksave.json!")

##### Func: Get Amount of Trees

In [ ]:
def getNumTrees(model):
    dump_list = model.get_booster().get_dump()
    num_trees = len(dump_list)
    return num_trees

##### Func: Get Amount of Splits

In [ ]:
def getNumSplits(model):
    trees_strings = model.get_booster().get_dump(dump_format='text')
    total_splits = 0
    for tree_string in trees_strings:
        n_nodes = len(tree_string.split('\n')) - 1
        n_leaves = tree_string.count('leaf')
        total_splits += n_nodes - n_leaves
    return total_splits

##### RFE

##### Main

In [ ]:
# importData()
# trainQuicksave(6)
model = XGBClassifier()
model.load_model("quicksave.json")
print(f'Number of trees: {getNumTrees(model)}')
print(f'Number of splits: {getNumSplits(model)}')
print(accuracy_score(ytest, model.predict(xtest)))

